# LeetCode 1954 — Minimum Garden Perimeter to Collect Enough Apples bằng Python

Link đề: https://leetcode.com/problems/minimum-garden-perimeter-to-collect-enough-apples/ · Độ khó: **Medium** · Chủ đề: Math, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def minimumPerimeter(self, neededApples: int) -> int:
        ...
```

Notebook đi từ **đếm táo từng ô** trong mảnh đất (vét cạn) → **cộng từng "vành" vuông** với công thức `12k²` → **công thức đóng** `2n(n+1)(2n+1)` + vòng lặp → **tìm kiếm nhị phân** trên `n` → **ước lượng căn bậc ba** rồi chỉnh lại, `O(1)`.

## 1. Mục tiêu học

1. Mô hình hóa mảnh đất: hình vuông tâm `(0, 0)`, cạnh song song trục, đỉnh nguyên ⇒ cạnh **chẵn** `2n`, chu vi `8n`.
2. Tự chứng minh số táo trên **vành thứ `k`** (các ô có `max(|x|, |y|) = k`) là `12k²`, và tổng tới vành `n` là `2n(n+1)(2n+1)`.
3. Dùng hàm **đơn điệu** của `n` để tìm `n` nhỏ nhất: duyệt tuyến tính, tìm kiếm nhị phân, hoặc ước lượng bằng căn bậc ba.
4. Cẩn thận với số lớn (`10¹⁵`) và sai số **số thực** khi dùng `**(1/3)`.
5. Kiểm thử bằng vét cạn đếm ô với `neededApples` nhỏ và đối chiếu các cách trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Một khu vườn là lưới 2 chiều vô hạn; tại **mỗi tọa độ nguyên** `(i, j)` có một cây táo với `|i| + |j|` quả.

Bạn mua một mảnh đất **hình vuông**, cạnh **song song trục tọa độ**, **tâm tại `(0, 0)`**. Cho `neededApples`, trả về **chu vi nhỏ nhất** của mảnh đất sao cho có **ít nhất** `neededApples` quả táo **bên trong hoặc trên biên**.

**Ví dụ 1:** `neededApples = 1` → `8`. Hình vuông cạnh 1 chỉ chứa điểm nguyên `(0, 0)` — 0 quả; hình vuông cạnh 2 chứa 12 quả (đủ); chu vi `2 · 4 = 8`.

**Ví dụ 2:** `neededApples = 13` → `16`.

**Ví dụ 3:** `neededApples = 1000000000` → `5040`.

**Ràng buộc:** `1 ≤ neededApples ≤ 10¹⁵`.

In [1]:
import math
import random
import time
from typing import List

examples = [(1, 8), (13, 16), (10**9, 5040)]

def apples_in_square(n):
    """Đếm trực tiếp số táo trong hình vuông [-n, n] x [-n, n] (cạnh 2n)."""
    return sum(abs(i) + abs(j) for i in range(-n, n + 1) for j in range(-n, n + 1))

for n in range(1, 5):
    print(f"cạnh {2 * n} (n={n}): {apples_in_square(n)} quả, chu vi {8 * n}")

cạnh 2 (n=1): 12 quả, chu vi 8
cạnh 4 (n=2): 60 quả, chu vi 16
cạnh 6 (n=3): 168 quả, chu vi 24
cạnh 8 (n=4): 360 quả, chu vi 32


Với `neededApples = 13`: cạnh 2 chỉ có 12 quả (thiếu), cạnh 4 có 60 quả (đủ) ⇒ chu vi `16`. Khớp ví dụ 2.

## 3. Kiến thức nền

### 3.1 Vì sao cạnh phải chẵn?

Hình vuông tâm `(0, 0)`, cạnh song song trục, cạnh dài `s` có các cạnh tại `x = ±s/2`, `y = ±s/2`. Muốn các ô nguyên trên biên được tính, và để so sánh "cạnh nhỏ nhất", ta chỉ cần xét `s/2 = n` nguyên: hình vuông cạnh `s` với `2n ≤ s < 2n + 2` chứa **cùng** tập điểm nguyên với hình cạnh `2n`, nên cạnh tối ưu luôn là số chẵn `2n`, chu vi `4 · 2n = 8n`.

In [2]:
def points_in_square_side(s):
    half = s / 2
    return [(i, j) for i in range(-3, 4) for j in range(-3, 4) if abs(i) <= half and abs(j) <= half]

for s in [1, 2, 3, 4]:
    pts = points_in_square_side(s)
    print(f"cạnh {s}: {len(pts)} điểm nguyên, táo = {sum(abs(i) + abs(j) for i, j in pts)}")

cạnh 1: 1 điểm nguyên, táo = 0
cạnh 2: 9 điểm nguyên, táo = 12
cạnh 3: 9 điểm nguyên, táo = 12
cạnh 4: 25 điểm nguyên, táo = 60


### 3.2 Tổng các bình phương

`1² + 2² + … + n² = n(n + 1)(2n + 1) / 6` — dùng để cộng các vành.

In [3]:
for n in [1, 2, 3, 10]:
    print(n, sum(k * k for k in range(1, n + 1)), n * (n + 1) * (2 * n + 1) // 6)

1 1 1
2 5 5
3 14 14
10 385 385


### 3.3 Tìm kiếm nhị phân "giá trị nhỏ nhất thỏa điều kiện"

Nếu `f(n)` **tăng** theo `n`, tìm `n` nhỏ nhất với `f(n) ≥ need` trong `O(log R)` lần tính `f`.

In [4]:
def first_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid): hi = mid
        else: lo = mid + 1
    return lo

print(first_true(1, 10**6, lambda n: n * n >= 12345))

112


### 3.4 Căn bậc ba và sai số số thực

`x ** (1/3)` cho số thực **gần đúng**; với số lớn có thể lệch 1. Luôn chỉnh lại bằng phép so sánh số nguyên.

In [5]:
for x in [27, 64, 10**15, (10**5) ** 3]:
    r = x ** (1 / 3)
    print(f"{x}: x**(1/3) = {r!r}, int = {int(r)}, đúng = {round(r)}")

27: x**(1/3) = 3.0, int = 3, đúng = 3
64: x**(1/3) = 3.9999999999999996, int = 3, đúng = 4
1000000000000000: x**(1/3) = 99999.99999999994, int = 99999, đúng = 100000
1000000000000000: x**(1/3) = 99999.99999999994, int = 99999, đúng = 100000


## 4. Ý tưởng nền tảng / trực giác

Gọi **vành thứ `k`** là các ô có `max(|x|, |y|) = k` (biên của hình vuông `[-k, k]²`). Hình vuông cạnh `2n` = vành `0, 1, …, n`.

Tính tay vành 1: 8 ô `(±1, 0), (0, ±1)` (mỗi ô 1 quả) và `(±1, ±1)` (mỗi ô 2 quả) ⇒ `4·1 + 4·2 = 12`.

Vành `k` tổng quát: xét cạnh phải `x = k`, `y ∈ [-k, k]`: táo `Σ_y (k + |y|) = (2k + 1)k + k(k + 1) = 3k² + 2k`... Cộng bốn cạnh nhưng 4 góc bị đếm hai lần (mỗi góc `2k` quả): `4(3k² + 2k) − 4·2k = 12k²`.

⇒ tổng tới vành `n`: `Σ_{k=1}^{n} 12k² = 12 · n(n+1)(2n+1)/6 = 2n(n+1)(2n+1)`.

In [6]:
def ring(k):
    return sum(abs(i) + abs(j) for i in range(-k, k + 1) for j in range(-k, k + 1) if max(abs(i), abs(j)) == k)

for k in range(1, 6):
    print(f"vành {k}: đếm = {ring(k):>4}, 12k² = {12 * k * k:>4} | tổng tới n={k}: đếm = {apples_in_square(k):>5}, 2n(n+1)(2n+1) = {2 * k * (k + 1) * (2 * k + 1):>5}")

vành 1: đếm =   12, 12k² =   12 | tổng tới n=1: đếm =    12, 2n(n+1)(2n+1) =    12
vành 2: đếm =   48, 12k² =   48 | tổng tới n=2: đếm =    60, 2n(n+1)(2n+1) =    60
vành 3: đếm =  108, 12k² =  108 | tổng tới n=3: đếm =   168, 2n(n+1)(2n+1) =   168
vành 4: đếm =  192, 12k² =  192 | tổng tới n=4: đếm =   360, 2n(n+1)(2n+1) =   360
vành 5: đếm =  300, 12k² =  300 | tổng tới n=5: đếm =   660, 2n(n+1)(2n+1) =   660


In bảng số táo của từng ô trong hình vuông `n = 2` (cạnh 4), đánh dấu vành của mỗi ô. Đây là cách "nhìn" công thức: vành 1 là khung `3×3` bỏ tâm, vành 2 là khung ngoài `5×5`.

In [7]:
n = 2
for y in range(n, -n - 1, -1):
    row = []
    for x in range(-n, n + 1):
        row.append(f"{abs(x) + abs(y)}({max(abs(x), abs(y))})")
    print("  ".join(f"{c:>5}" for c in row))
print("định dạng: số_táo(vành). Tổng vành 1 =", ring(1), ", vành 2 =", ring(2), ", tổng =", apples_in_square(2))

 4(2)   3(2)   2(2)   3(2)   4(2)
 3(2)   2(1)   1(1)   2(1)   3(2)
 2(2)   1(1)   0(0)   1(1)   2(2)
 3(2)   2(1)   1(1)   2(1)   3(2)
 4(2)   3(2)   2(2)   3(2)   4(2)
định dạng: số_táo(vành). Tổng vành 1 = 12 , vành 2 = 48 , tổng = 60


Chứng minh chi tiết `12k²` theo từng cạnh của vành `k` (cạnh phải `x = k`, `y` chạy từ `−k` tới `k`):

`Σ_{y=−k}^{k} (k + |y|) = (2k + 1)·k + 2·(1 + 2 + … + k) = (2k + 1)k + k(k + 1) = 3k² + 2k`.

Bốn cạnh cho `4(3k² + 2k) = 12k² + 8k`, nhưng bốn góc `(±k, ±k)` (mỗi góc `2k` quả) bị đếm hai lần ⇒ trừ `4 · 2k = 8k` ⇒ `12k²`.

In [8]:
for k in range(1, 7):
    side = sum(k + abs(y) for y in range(-k, k + 1))
    print(f"k={k}: một cạnh = {side} (= 3k²+2k = {3 * k * k + 2 * k}), bốn cạnh - góc = {4 * side - 8 * k}, 12k² = {12 * k * k}")

k=1: một cạnh = 5 (= 3k²+2k = 5), bốn cạnh - góc = 12, 12k² = 12
k=2: một cạnh = 16 (= 3k²+2k = 16), bốn cạnh - góc = 48, 12k² = 48
k=3: một cạnh = 33 (= 3k²+2k = 33), bốn cạnh - góc = 108, 12k² = 108
k=4: một cạnh = 56 (= 3k²+2k = 56), bốn cạnh - góc = 192, 12k² = 192
k=5: một cạnh = 85 (= 3k²+2k = 85), bốn cạnh - góc = 300, 12k² = 300
k=6: một cạnh = 120 (= 3k²+2k = 120), bốn cạnh - góc = 432, 12k² = 432


## 5. Cách 1 — Brute Force / baseline: tăng `n`, đếm lại mọi ô

**Ý tưởng:** thử `n = 1, 2, …`; mỗi lần đếm lại toàn bộ táo trong hình vuông `[-n, n]²`; dừng khi đủ.

In [9]:
def min_perimeter_brute(needed):
    n = 1
    while apples_in_square(n) < needed:
        n += 1
    return 8 * n

for need, exp in examples[:2]:
    print(min_perimeter_brute(need), "(kỳ vọng", exp, ")")

8 (kỳ vọng 8 )
16 (kỳ vọng 16 )


**Độ phức tạp:** để đạt `need` cần `n ≈ (need/4)^(1/3)`; mỗi lần đếm `O(n²)` ô ⇒ tổng `O(n³) = O(need)`. Với `need = 10¹⁵` hoàn toàn không khả thi (ví dụ 3 cũng đã cần `n = 630`, tức `~10⁸` phép tính — ta không chạy).

**Nút thắt:** đếm lại từ đầu; trong khi chỉ cần cộng thêm **vành mới** — và vành mới có công thức `12n²`.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Gia tăng:** `S(n) = S(n − 1) + 12n²` ⇒ duyệt `n` tăng dần, mỗi bước `O(1)`; cần `~(need/4)^(1/3) ≤ 63 000` bước với `need ≤ 10¹⁵`.
2. **Công thức đóng:** `S(n) = 2n(n+1)(2n+1)` (bậc ba, tăng nghiêm ngặt).
3. **Đơn điệu ⇒ tìm kiếm nhị phân:** `n* = min{n ≥ 1 : S(n) ≥ need}`.
4. **Ước lượng:** `S(n) ≈ 4n³` ⇒ `n ≈ (need/4)^(1/3)`; tính gần đúng rồi **chỉnh lên/xuống** bằng so sánh số nguyên.

**Quy tắc:** **nếu** `S(n) ≥ need` **thì** `n` khả thi và mọi `n' > n` cũng khả thi; **ngược lại** mọi `n' < n` đều không khả thi.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cộng dồn từng vành `12k²`

In [10]:
def min_perimeter_rings(needed):
    total, n = 0, 0
    while total < needed:
        n += 1
        total += 12 * n * n
    return 8 * n

for need, exp in examples:
    print(min_perimeter_rings(need), "(kỳ vọng", exp, ")")

8 (kỳ vọng 8 )
16 (kỳ vọng 16 )
5040 (kỳ vọng 5040 )


**Độ phức tạp:** `O(n*) = O(need^(1/3))` ≈ `6.3·10⁴` bước khi `need = 10¹⁵`; bộ nhớ `O(1)`.

### 7.2 Cách 3 — Công thức đóng + vòng lặp tuyến tính

In [11]:
def S(n):
    return 2 * n * (n + 1) * (2 * n + 1)

def min_perimeter_formula_loop(needed):
    n = 1
    while S(n) < needed:
        n += 1
    return 8 * n

class SolutionLoop:
    def minimumPerimeter(self, neededApples: int) -> int:
        n = 1
        while 2 * n * (n + 1) * (2 * n + 1) < neededApples:
            n += 1
        return 8 * n

for need, exp in examples:
    print(min_perimeter_formula_loop(need), SolutionLoop().minimumPerimeter(need), "(kỳ vọng", exp, ")")

8 8 (kỳ vọng 8 )
16 16 (kỳ vọng 16 )
5040 5040 (kỳ vọng 5040 )


**Độ phức tạp:** như cách 2, `O(need^(1/3))`.

### 7.3 Cách 4 — Tìm kiếm nhị phân trên `n`

Cận trên an toàn: `S(n) ≥ 4n³`, nên với `n = 10⁵` có `S ≥ 4·10¹⁵ ≥ need`. Tìm `n` nhỏ nhất trong `[1, 10⁵]`.

In [12]:
def min_perimeter_binary(needed):
    lo, hi = 1, 10**5
    while lo < hi:
        mid = (lo + hi) // 2
        if S(mid) >= needed:
            hi = mid
        else:
            lo = mid + 1
    return 8 * lo

for need, exp in examples:
    print(min_perimeter_binary(need), "(kỳ vọng", exp, ")")

8 (kỳ vọng 8 )
16 (kỳ vọng 16 )
5040 (kỳ vọng 5040 )


**Độ phức tạp:** `O(log 10⁵) ≈ 17` lần tính `S` ⇒ `O(log need)`; bộ nhớ `O(1)`.

### 7.4 Cách 5 — Ước lượng bằng căn bậc ba rồi chỉnh

`n₀ = int((need / 4) ** (1/3))`; sau đó giảm `n` khi `S(n − 1) ≥ need` và tăng khi `S(n) < need`. Vì `n₀` lệch rất ít, chỉ vài bước chỉnh.

In [13]:
def min_perimeter_cbrt(needed):
    n = max(1, int((needed / 4) ** (1 / 3)))
    while n > 1 and S(n - 1) >= needed:
        n -= 1
    while S(n) < needed:
        n += 1
    return 8 * n

for need, exp in examples:
    print(min_perimeter_cbrt(need), "(kỳ vọng", exp, ")")

8 (kỳ vọng 8 )
16 (kỳ vọng 16 )
5040 (kỳ vọng 5040 )


**Độ phức tạp:** `O(1)` (một phép lũy thừa số thực + vài bước chỉnh). Cần hai vòng chỉnh để **không phụ thuộc** vào độ chính xác số thực.

## 8. TRACE từng bước

Trace cách 2 với `needed = 200`:

| n | vành 12n² | tổng | đủ 200? |
|---|---|---|---|
| 1 | 12 | 12 | |
| 2 | 48 | 60 | |
| 3 | 108 | 168 | |
| 4 | 192 | 360 | ✔ ⇒ chu vi 32 |

In [14]:
def trace_rings(needed):
    total, n = 0, 0
    while total < needed:
        n += 1
        total += 12 * n * n
        print(f"  n={n}: vành = 12*{n}^2 = {12 * n * n:>6}, tổng = {total:>7} {'>=' if total >= needed else '<'} {needed}")
    print(f"  => n = {n}, chu vi = {8 * n}")
    return 8 * n

trace_rings(200)
trace_rings(13)

  n=1: vành = 12*1^2 =     12, tổng =      12 < 200
  n=2: vành = 12*2^2 =     48, tổng =      60 < 200
  n=3: vành = 12*3^2 =    108, tổng =     168 < 200
  n=4: vành = 12*4^2 =    192, tổng =     360 >= 200
  => n = 4, chu vi = 32
  n=1: vành = 12*1^2 =     12, tổng =      12 < 13
  n=2: vành = 12*2^2 =     48, tổng =      60 >= 13
  => n = 2, chu vi = 16


16

Trace cách 4 (tìm kiếm nhị phân) với `needed = 10⁹`:

In [15]:
def trace_binary(needed):
    lo, hi = 1, 10**5
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        ok = S(mid) >= needed
        step += 1
        print(f"  bước {step:>2}: lo={lo:>6}, hi={hi:>6}, mid={mid:>6}, S(mid)={S(mid):>20} -> {'đủ, hi=mid' if ok else 'thiếu, lo=mid+1'}")
        if ok: hi = mid
        else: lo = mid + 1
    print(f"  => n = {lo}, chu vi = {8 * lo}")
    return 8 * lo

trace_binary(10**9)

  bước  1: lo=     1, hi=100000, mid= 50000, S(mid)=     500015000100000 -> đủ, hi=mid
  bước  2: lo=     1, hi= 50000, mid= 25000, S(mid)=      62503750050000 -> đủ, hi=mid
  bước  3: lo=     1, hi= 25000, mid= 12500, S(mid)=       7813437525000 -> đủ, hi=mid
  bước  4: lo=     1, hi= 12500, mid=  6250, S(mid)=        976796887500 -> đủ, hi=mid
  bước  5: lo=     1, hi=  6250, mid=  3125, S(mid)=        122128912500 -> đủ, hi=mid
  bước  6: lo=     1, hi=  3125, mid=  1563, S(mid)=         15288103128 -> đủ, hi=mid
  bước  7: lo=     1, hi=  1563, mid=   782, S(mid)=          1916517780 -> đủ, hi=mid
  bước  8: lo=     1, hi=   782, mid=   391, S(mid)=           240023952 -> thiếu, lo=mid+1
  bước  9: lo=   392, hi=   782, mid=   587, S(mid)=           811116600 -> thiếu, lo=mid+1
  bước 10: lo=   588, hi=   782, mid=   685, S(mid)=          1288493220 -> đủ, hi=mid
  bước 11: lo=   588, hi=   685, mid=   636, S(mid)=          1031466072 -> đủ, hi=mid
  bước 12: lo=   588, hi=   636, 

5040

Trace cách 5 (căn bậc ba + chỉnh) cho vài giá trị — để thấy ước lượng số thực lệch bao nhiêu và chỉnh mấy bước:

In [16]:
def trace_cbrt(needed):
    n0 = max(1, int((needed / 4) ** (1 / 3)))
    n = n0; down = up = 0
    while n > 1 and S(n - 1) >= needed:
        n -= 1; down += 1
    while S(n) < needed:
        n += 1; up += 1
    print(f"  need={needed:>17}: ước lượng n0={n0:>6}, chỉnh xuống {down}, chỉnh lên {up} -> n={n}, chu vi={8 * n}")
    return 8 * n

for need in [1, 13, 10**9, 10**15, S(1000), S(1000) + 1, S(62996)]:
    trace_cbrt(need)

  need=                1: ước lượng n0=     1, chỉnh xuống 0, chỉnh lên 0 -> n=1, chu vi=8
  need=               13: ước lượng n0=     1, chỉnh xuống 0, chỉnh lên 1 -> n=2, chu vi=16
  need=       1000000000: ước lượng n0=   629, chỉnh xuống 0, chỉnh lên 1 -> n=630, chu vi=5040
  need= 1000000000000000: ước lượng n0= 62996, chỉnh xuống 0, chỉnh lên 0 -> n=62996, chu vi=503968
  need=       4006002000: ước lượng n0=  1000, chỉnh xuống 0, chỉnh lên 0 -> n=1000, chu vi=8000
  need=       4006002001: ước lượng n0=  1000, chỉnh xuống 0, chỉnh lên 1 -> n=1001, chu vi=8008
  need= 1000021311197832: ước lượng n0= 62996, chỉnh xuống 0, chỉnh lên 0 -> n=62996, chu vi=503968


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def minimumPerimeter(self, neededApples: int) -> int:
        lo, hi = 1, 10**5                     # S(10^5) >= 4*10^15 >= neededApples
        while lo < hi:
            mid = (lo + hi) // 2
            if 2 * mid * (mid + 1) * (2 * mid + 1) >= neededApples:
                hi = mid
            else:
                lo = mid + 1
        return 8 * lo

for need, exp in examples:
    assert Solution().minimumPerimeter(need) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Chỉ cần xét cạnh `2n`.** Mọi hình vuông hợp lệ cạnh `s` chứa đúng các điểm nguyên `|x|, |y| ≤ ⌊s/2⌋`; với `n = ⌊s/2⌋` thì hình cạnh `2n ≤ s` chứa cùng tập điểm và có chu vi không lớn hơn. Nếu `n = 0` thì chỉ có điểm `(0,0)` với 0 quả — không đủ vì `need ≥ 1`. ⇒ đáp án `= 8n*` với `n*` nhỏ nhất mà `S(n*) ≥ need`.
2. **Công thức `S(n)`.** Vành `k ≥ 1` có `12k²` quả (mục 4: bốn cạnh, trừ bốn góc đếm hai lần), nên `S(n) = 12 Σk² = 2n(n+1)(2n+1)`. Đã kiểm bằng đếm trực tiếp với `n ≤ 5`, và ở mục 12 với `n ≤ 40`.
3. **Đơn điệu.** `S(n + 1) − S(n) = 12(n + 1)² > 0` ⇒ tìm kiếm nhị phân / duyệt tuyến tính trả về đúng `n*`.
4. **Cận trên.** `S(10⁵) > 4·10¹⁵ ≥ 10¹⁵ ≥ need` nên `n* ≤ 10⁵`.

## 11. Phân tích Time / Space Complexity

Gọi `N = neededApples`, `n* ≈ (N/4)^(1/3)` (tối đa `≈ 63 000`).

| Cách | Time | Space |
|---|---|---|
| 1. Đếm lại mọi ô | `O(n*³) = O(N)` | `O(1)` |
| 2. Cộng vành | `O(n*) = O(N^(1/3))` | `O(1)` |
| 3. Công thức + vòng lặp | `O(N^(1/3))` | `O(1)` |
| 4. Tìm kiếm nhị phân | `O(log N)` | `O(1)` |
| 5. Căn bậc ba + chỉnh | `O(1)` | `O(1)` |

Trong Python số nguyên không tràn; trong C++/Java cần `long long`/`long` vì `S(n)` tới `~4·10¹⁵`.

In [18]:
N = 10**15
for name, fn in [("rings", min_perimeter_rings), ("formula_loop", min_perimeter_formula_loop), ("binary", min_perimeter_binary), ("cbrt", min_perimeter_cbrt), ("Solution", Solution().minimumPerimeter)]:
    t0 = time.perf_counter(); r = fn(N); dt = time.perf_counter() - t0
    print(f"{name:<13} N=10^15: {r}  {dt * 1000:8.3f} ms")

rings         N=10^15: 503968     5.914 ms
formula_loop  N=10^15: 503968    10.202 ms
binary        N=10^15: 503968     0.007 ms
cbrt          N=10^15: 503968     0.005 ms
Solution      N=10^15: 503968     0.004 ms


## 12. Bộ test

1. Ví dụ đề.
2. Kiểm công thức `S(n)` bằng đếm ô với `n ≤ 40`.
3. Đối chiếu brute force với mọi `need ≤ 2000` (brute chạy nhanh vì `n` nhỏ).
4. **Ngay tại biên**: `need = S(n)` và `need = S(n) + 1` với nhiều `n`, kể cả lớn (gần `10¹⁵`) — chỗ hay sai lệch một.
5. Ngẫu nhiên lớn có seed: các cách nhanh khớp nhau.

In [19]:
ALL = {
    "rings": min_perimeter_rings,
    "formula_loop": min_perimeter_formula_loop,
    "binary": min_perimeter_binary,
    "cbrt": min_perimeter_cbrt,
    "SolLoop": lambda x: SolutionLoop().minimumPerimeter(x),
    "Solution": lambda x: Solution().minimumPerimeter(x),
}

def check(need, exp=None, brute=False):
    ref = min_perimeter_brute(need) if brute else min_perimeter_rings(need)
    for name, fn in ALL.items():
        assert fn(need) == ref, (name, need, fn(need), ref)
    if exp is not None:
        assert ref == exp
    return ref

for need, exp in examples:
    check(need, exp)
print("Ví dụ đề: OK")

for n in range(1, 41):
    assert apples_in_square(n) == S(n)
print("Công thức S(n) khớp đếm ô với n <= 40: OK")

for need in range(1, 2001):
    check(need, brute=True)
print("Mọi need <= 2000 so với brute force: OK")

for n in list(range(1, 200)) + [62995, 62996, 62999, 63000, 99999]:
    if S(n) <= 10**15:
        check(S(n), 8 * n)               # vừa đủ ở n
        check(S(n) + 1, 8 * (n + 1))     # thiếu 1 -> cần thêm vành
    check(max(1, S(n) - 1), 8 * n)
print("Biên S(n), S(n)+1, S(n)-1: OK")

rnd = random.Random(1954)
for _ in range(300):
    need = rnd.randint(1, 10**15)
    a = Solution().minimumPerimeter(need)
    assert a == min_perimeter_cbrt(need) == min_perimeter_binary(need)
    n = a // 8
    assert S(n) >= need and (n == 1 or S(n - 1) < need)
print("300 need ngẫu nhiên tới 10^15: OK")

Ví dụ đề: OK
Công thức S(n) khớp đếm ô với n <= 40: OK
Mọi need <= 2000 so với brute force: OK


Biên S(n), S(n)+1, S(n)-1: OK
300 need ngẫu nhiên tới 10^15: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chu vi `4n` (nhầm `n` là cạnh)

In [20]:
def min_perimeter_bug_4n(needed):
    n = 1
    while S(n) < needed:
        n += 1
    return 4 * n                         # SAI: cạnh là 2n nên chu vi là 8n

print("need=1: sai =", min_perimeter_bug_4n(1), "| đúng =", Solution().minimumPerimeter(1))

need=1: sai = 4 | đúng = 8


**Vì sao sai:** `n` là **nửa cạnh**; cạnh `2n`, chu vi `4 · 2n = 8n`. **Sửa:** `return 8 * n`.

### Lỗi 2 — Công thức vành sai (đếm trùng góc)

In [21]:
def min_perimeter_bug_ring(needed):
    total, n = 0, 0
    while total < needed:
        n += 1
        total += 4 * (3 * n * n + 2 * n)   # SAI: quên trừ 4 góc bị đếm 2 lần
    return 8 * n

rnd = random.Random(2)
found = next(need for need in range(1, 5000) if min_perimeter_bug_ring(need) != Solution().minimumPerimeter(need))
print("phản ví dụ need =", found, "| sai:", min_perimeter_bug_ring(found), "| đúng:", Solution().minimumPerimeter(found))

phản ví dụ need = 13 | sai: 8 | đúng: 16


**Vì sao sai:** mỗi cạnh của vành có `2k + 1` ô, bốn cạnh có `8k + 4` ô nhưng vành chỉ có `8k` ô — bốn góc (mỗi góc `2k` quả) bị cộng hai lần. **Sửa:** `12k²` (đếm trực tiếp để kiểm như mục 4).

### Lỗi 3 — Dùng căn bậc ba số thực mà không chỉnh

In [22]:
def min_perimeter_bug_float(needed):
    n = math.ceil((needed / 4) ** (1 / 3))       # SAI: S(n) không đúng bằng 4n^3, và số thực có sai số
    return 8 * n

bad = [need for need in range(1, 3000) if min_perimeter_bug_float(need) != Solution().minimumPerimeter(need)]
print("số need <= 3000 bị sai:", len(bad), "| ví dụ:", bad[:5], "->", [(min_perimeter_bug_float(x), Solution().minimumPerimeter(x)) for x in bad[:3]])

số need <= 3000 bị sai: 1379 | ví dụ: [5, 6, 7, 8, 9] -> [(16, 8), (16, 8), (16, 8)]


**Vì sao sai:** `S(n) = 4n³ + 6n² + 2n > 4n³`, nên `ceil((need/4)^(1/3))` thường **thừa** một; cộng thêm sai số dấu phẩy động. **Sửa:** dùng số thực chỉ để **ước lượng**, rồi chỉnh bằng so sánh số nguyên (cách 5), hoặc tìm kiếm nhị phân.

### Lỗi 4 — So sánh `>` thay vì `≥` (khi vừa đủ)

In [23]:
def min_perimeter_bug_strict(needed):
    n = 1
    while not (S(n) > needed):            # SAI: "ít nhất" nghĩa là >=
        n += 1
    return 8 * n

print("need=12: sai =", min_perimeter_bug_strict(12), "| đúng =", Solution().minimumPerimeter(12))

need=12: sai = 16 | đúng = 8


**Vì sao sai:** đề yêu cầu **ít nhất** `neededApples` quả; khi `S(n) == need` thì `n` đã đủ. **Sửa:** điều kiện `S(n) >= need`.

### Lỗi 5 — Cận trên tìm kiếm nhị phân quá nhỏ, hoặc tràn số 32-bit (khi chuyển sang C++/Java)

In [24]:
def min_perimeter_bug_small_hi(needed):
    lo, hi = 1, 1000                       # SAI: S(1000) ≈ 4·10^9 < 10^15
    while lo < hi:
        mid = (lo + hi) // 2
        if S(mid) >= needed: hi = mid
        else: lo = mid + 1
    return 8 * lo

def S_int32(n):
    v = (2 * n * (n + 1) * (2 * n + 1)) & 0xFFFFFFFF   # mô phỏng tràn số 32-bit không dấu...
    return v - (1 << 32) if v >= (1 << 31) else v      # ...rồi đổi sang có dấu

print("cận nhỏ, need=10^15: sai =", min_perimeter_bug_small_hi(10**15), "| đúng =", Solution().minimumPerimeter(10**15))
print("S(1000) đúng =", S(1000), "| tính bằng int32 =", S_int32(1000))

cận nhỏ, need=10^15: sai = 8000 | đúng = 503968
S(1000) đúng = 4006002000 | tính bằng int32 = -288965296


**Vì sao sai:** với cận `1000` tìm kiếm nhị phân luôn trả `1000` khi `need` lớn — không còn chỗ để đạt điều kiện. Trong ngôn ngữ có số nguyên cố định, `S(n)` vượt `2³¹` từ `n ≈ 650` ⇒ phải dùng 64-bit. **Sửa:** cận `10⁵` (vì `S(10⁵) > 4·10¹⁵`) và dùng `long`/`long long`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Lưới vô hạn + giá trị `|i| + |j|` + hình vuông tâm gốc ⇒ chia theo **vành** `max(|x|, |y|) = k`.
- Tìm công thức bằng cách **tính tay vài giá trị nhỏ** (`12, 60, 168, 360, …`) rồi đoán/kiểm (`12k²`, tổng bình phương).
- "Nhỏ nhất sao cho tổng ≥ need" với tổng **đơn điệu** ⇒ tìm kiếm nhị phân; nếu số bước tuyến tính nhỏ (`≤ 10⁵`) thì duyệt cũng được.
- Số thực (`** (1/3)`, `sqrt`) chỉ dùng để **ước lượng**, luôn chỉnh lại bằng số nguyên (`math.isqrt` cho căn bậc hai).
- Viết hàm đếm brute force để **kiểm công thức** trước khi tin nó.

## 15. Biến thể của bài

1. **Mỗi cây có `max(|i|, |j|)` quả** (thay vì `|i| + |j|`): vành `k` có `8k` ô, mỗi ô `k` quả ⇒ `8k²`.
2. **Mảnh đất hình thoi** `|x| + |y| ≤ n`: lớp `|x| + |y| = k` có `4k` ô, mỗi ô `k` quả ⇒ `4k²`.
3. **Cho chu vi, hỏi số táo:** `S(P/8)`.
4. **Hình chữ nhật `[-a, a] × [-b, b]`:** tổng theo hai chiều tách rời: `Σ_{i,j} (|i| + |j|) = (2b + 1)·a(a + 1) + (2a + 1)·b(b + 1)`.
5. **LeetCode 69 — Sqrt(x), 367 — Valid Perfect Square:** tìm kiếm nhị phân trên đáp án cho hàm đơn điệu.

In [25]:
def apples_rect(a, b):
    return (2 * b + 1) * a * (a + 1) + (2 * a + 1) * b * (b + 1)

def apples_rect_brute(a, b):
    return sum(abs(i) + abs(j) for i in range(-a, a + 1) for j in range(-b, b + 1))

def apples_diamond(n):
    return sum(4 * k * k for k in range(1, n + 1))

def apples_diamond_brute(n):
    return sum(abs(i) + abs(j) for i in range(-n, n + 1) for j in range(-n, n + 1) if abs(i) + abs(j) <= n)

assert all(apples_rect(a, b) == apples_rect_brute(a, b) for a in range(6) for b in range(6))
assert all(apples_diamond(n) == apples_diamond_brute(n) for n in range(10))
print("Hình chữ nhật và hình thoi: công thức khớp đếm ô. Ví dụ rect(2,3) =", apples_rect(2, 3), ", diamond(3) =", apples_diamond(3))

Hình chữ nhật và hình thoi: công thức khớp đếm ô. Ví dụ rect(2,3) = 102 , diamond(3) = 56


Biến thể 1 (mỗi cây có `max(|i|, |j|)` quả): vành `k` có `8k` ô, mỗi ô `k` quả ⇒ `8k²`, tổng tới `n` là `4n(n+1)(2n+1)/3`. Kiểm bằng đếm ô:

In [26]:
def apples_max_metric(n):
    return 4 * n * (n + 1) * (2 * n + 1) // 3

def apples_max_metric_brute(n):
    return sum(max(abs(i), abs(j)) for i in range(-n, n + 1) for j in range(-n, n + 1))

assert all(apples_max_metric(n) == apples_max_metric_brute(n) for n in range(30))
print("max-metric:", [apples_max_metric(n) for n in range(1, 6)])

max-metric: [8, 40, 112, 240, 440]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm ô | tăng `n`, đếm lại | `O(N)` | `O(1)` | kiểm thử, `N` nhỏ |
| 2. Cộng vành | `+12k²` | `O(N^(1/3))` | `O(1)` | khi chưa có công thức đóng |
| 3. Công thức + duyệt | `S(n) = 2n(n+1)(2n+1)` | `O(N^(1/3))` | `O(1)` | đơn giản, đủ nhanh |
| 4. Tìm kiếm nhị phân | `S` đơn điệu | `O(log N)` | `O(1)` | **nộp bài**, chắc chắn |
| 5. Căn bậc ba | ước lượng + chỉnh | `O(1)` | `O(1)` | nhanh nhất, cần cẩn thận |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minimumPerimeter(60)`, `(61)`, `(168)`?

**Bài 2 (tính tay):** Đếm số táo trên vành `k = 2` bằng cách liệt kê ô theo từng giá trị `|x| + |y|`, kiểm với `12k²`.

**Bài 3 (điền TODO):** Hoàn thiện `max_apples_with_perimeter(P)` trả về số táo **tối đa** thu được với chu vi **không vượt quá** `P` (P nguyên dương).

**Bài 4 (tự viết lại):** Viết lại cách 4 mà không đặt cận trên cố định `10⁵`: bắt đầu `hi = 1` và **nhân đôi** cho tới khi `S(hi) ≥ need` (exponential search).

**Bài 5 (mini challenge):** Biến thể hình thoi: mảnh đất `|x| + |y| ≤ n` (n nguyên). Viết `min_diamond_n(need)` trả về `n` nhỏ nhất sao cho số táo `≥ need`, dùng công thức đóng + tìm kiếm nhị phân; kiểm bằng đếm ô.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def max_apples_with_perimeter(P):
    # TODO: n = P // 8 (nửa cạnh lớn nhất), trả S(n) (n = 0 -> 0)
    pass

# Bài 4 — TODO
def min_perimeter_exponential(needed):
    # TODO: hi = 1; while S(hi) < needed: hi *= 2; rồi tìm kiếm nhị phân trong [hi//2 + 1, hi]
    pass

# Bài 5 — TODO
def min_diamond_n(need):
    # TODO: D(n) = sum 4k^2 = 2n(n+1)(2n+1)/3; tìm n nhỏ nhất với D(n) >= need
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `S(1) = 12`, `S(2) = 60`, `S(3) = 168`.
- Mức 2: `60` vừa đủ ở `n = 2`; `61` cần `n = 3`.
- Mức 3: `16, 24, 24`.

**Bài 2**
- Mức 1: Vành 2 có 16 ô.
- Mức 2: Số ô có `|x|+|y| = 2, 3, 4` trên vành: 4, 8, 4.
- Mức 3: `4·2 + 8·3 + 4·4 = 48 = 12·4`.

**Bài 3**
- Mức 1: Chu vi `8n ≤ P`.
- Mức 2: `n = P // 8`.
- Mức 3: `P < 8` ⇒ 0 quả.

**Bài 4**
- Mức 1: Nhân đôi tới khi đủ.
- Mức 2: Đáp án nằm trong `(hi/2, hi]`.
- Mức 3: Số lần nhân đôi `O(log n*)`.

**Bài 5**
- Mức 1: Lớp `|x|+|y| = k` có `4k` ô, mỗi ô `k` quả.
- Mức 2: `D(n) = 4 Σ k² = 2n(n+1)(2n+1)/3`.
- Mức 3: `D` tăng nghiêm ngặt ⇒ tìm kiếm nhị phân.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
r1 = [Solution().minimumPerimeter(x) for x in (60, 61, 168)]
print("Bài 1:", r1)
assert r1 == [16, 24, 24]

# Bài 2
cells = [(i, j) for i in range(-2, 3) for j in range(-2, 3) if max(abs(i), abs(j)) == 2]
from collections import Counter
by_val = Counter(abs(i) + abs(j) for i, j in cells)
print("Bài 2:", len(cells), "ô; số ô theo |x|+|y|:", dict(sorted(by_val.items())), "-> tổng", sum(v * c for v, c in by_val.items()))
assert sum(v * c for v, c in by_val.items()) == 48

# Bài 3
def max_apples_with_perimeter_sol(P):
    n = P // 8
    return S(n) if n >= 1 else 0

for P in range(1, 400):
    n = P // 8
    assert max_apples_with_perimeter_sol(P) == (apples_in_square(n) if n else 0)
print("Bài 3 OK | P=20 ->", max_apples_with_perimeter_sol(20))

Bài 1: [16, 24, 24]
Bài 2: 16 ô; số ô theo |x|+|y|: {2: 4, 3: 8, 4: 4} -> tổng 48
Bài 3 OK | P=20 -> 60


In [29]:
# Bài 4
def min_perimeter_exponential_sol(needed):
    hi = 1
    while S(hi) < needed:
        hi *= 2
    lo = hi // 2 + 1 if hi > 1 else 1
    while lo < hi:
        mid = (lo + hi) // 2
        if S(mid) >= needed: hi = mid
        else: lo = mid + 1
    return 8 * lo

rnd = random.Random(4)
for _ in range(2000):
    need = rnd.choice([rnd.randint(1, 1000), rnd.randint(1, 10**15)])
    assert min_perimeter_exponential_sol(need) == Solution().minimumPerimeter(need)
print("Bài 4 OK")

# Bài 5
def D(n):
    return 2 * n * (n + 1) * (2 * n + 1) // 3

def min_diamond_n_sol(need):
    lo, hi = 1, 1
    while D(hi) < need:
        hi *= 2
    while lo < hi:
        mid = (lo + hi) // 2
        if D(mid) >= need: hi = mid
        else: lo = mid + 1
    return lo

for n in range(1, 25):
    assert D(n) == apples_diamond_brute(n)
for need in range(1, 3000):
    n = min_diamond_n_sol(need)
    assert D(n) >= need and (n == 1 or D(n - 1) < need)
print("Bài 5 OK | need=100 -> n =", min_diamond_n_sol(100))

Bài 4 OK
Bài 5 OK | need=100 -> n = 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "n nhỏ nhất với f(n) ≥ need" (f tăng):**

```python
lo, hi = 1, UPPER                 # chắc chắn f(UPPER) >= need
while lo < hi:
    mid = (lo + hi) // 2
    if f(mid) >= need: hi = mid
    else: lo = mid + 1
return lo
```

**Bài này:** `f(n) = 2n(n+1)(2n+1)`, `UPPER = 10⁵`, đáp án `8 · lo`.

**Checklist:**
- [ ] Chu vi `8n` (n là nửa cạnh)?
- [ ] Công thức vành `12k²` đã kiểm bằng đếm ô?
- [ ] Điều kiện `≥` (ít nhất)?
- [ ] Không tin tuyệt đối vào `** (1/3)` — có bước chỉnh số nguyên?
- [ ] Cận trên đủ lớn cho `need = 10¹⁵`?
- [ ] Chữ ký đúng: `def minimumPerimeter(self, neededApples: int) -> int`?